# Private API storage and quota probe
CPU only. Uses Kaggle built-in authentication through the official SDK. No training or credential creation.

In [ ]:
import hashlib
import json
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import kagglehub
from kagglehub.clients import build_kaggle_client
from kagglesdk.datasets.types.dataset_api_service import (
    ApiGetDatasetRequest,
    ApiGetDatasetStatusRequest,
)
from kagglesdk.datasets.types.dataset_enums import DatabundleVersionStatus
from kagglesdk.kernels.types.kernels_api_service import ApiGetAcceleratorQuotaStatisticsRequest

base = Path("/kaggle/working/hung-api-storage-probe-v1")
base.mkdir(exist_ok=False)
stage = base / "upload"
stage.mkdir()
handle = "hungluongduc/hung-e2e-api-storage-probe-20261027-v1"
with build_kaggle_client() as client:
    response = client.kernels.kernels_api_client.get_accelerator_quota_statistics(
        ApiGetAcceleratorQuotaStatisticsRequest()
    )
    gpu = response.gpu_quota
    assert gpu is not None
    assert all(v is not None for v in (gpu.time_used, gpu.time_reserved, gpu.total_time_allowed))
    quota = {
        "observed_at_utc": datetime.now(timezone.utc).isoformat(),
        "used_seconds": gpu.time_used.total_seconds(),
        "reserved_seconds": gpu.time_reserved.total_seconds(),
        "total_seconds": gpu.total_time_allowed.total_seconds(),
        "pay_to_scale_enabled": gpu.is_pay_to_scale_enabled,
    }
    assert not quota["pay_to_scale_enabled"], "Do not use paid scaling in this task"
    print(json.dumps({"quota": quota}), flush=True)
probe = {
    "schema": "hung-private-durable-storage-probe-v1",
    "code_sha": "44b8532959d76fb20691881539f991ca44983727",
    "scope": "Tiny control receipt only; no model execution, credentials or personal data",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
archive = stage / "probe.zip.bin"
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as stream:
    stream.writestr("probe.json", json.dumps(probe, sort_keys=True))
expected = hashlib.sha256(archive.read_bytes()).hexdigest()
started = time.perf_counter()
kagglehub.dataset_upload(
    handle, str(stage), version_notes="Private Development storage capability probe, version 1"
)
upload_seconds = time.perf_counter() - started
status_request = ApiGetDatasetStatusRequest()
status_request.owner_slug = "hungluongduc"
status_request.dataset_slug = handle.split("/")[1]
with build_kaggle_client() as client:
    for attempt in range(60):
        status = client.datasets.dataset_api_client.get_dataset_status(status_request).status
        if status == DatabundleVersionStatus.READY:
            break
        assert status not in (DatabundleVersionStatus.FAILED, DatabundleVersionStatus.DELETED), (
            status.name
        )
        time.sleep(10)
    else:
        raise TimeoutError("Dataset processing did not reach READY")
    request = ApiGetDatasetRequest()
    request.owner_slug = "hungluongduc"
    request.dataset_slug = handle.split("/")[1]
    dataset = client.datasets.dataset_api_client.get_dataset(request)
    assert dataset.is_private and dataset.current_version_number == 1, "Unexpected privacy/version"
readback_started = time.perf_counter()
remote = Path(
    kagglehub.dataset_download(handle + "/versions/1", path="probe.zip.bin", force_download=True)
)
if remote.is_dir():
    remote = remote / "probe.zip.bin"
assert remote.resolve() != archive.resolve()
assert hashlib.sha256(remote.read_bytes()).hexdigest() == expected
with zipfile.ZipFile(remote) as stream:
    assert stream.testzip() is None and stream.namelist() == ["probe.json"]
result = {
    "passed": True,
    "handle": handle,
    "version": 1,
    "is_private": dataset.is_private,
    "sha256": expected,
    "bytes": archive.stat().st_size,
    "crc": "passed",
    "upload_seconds": upload_seconds,
    "provider_readback_seconds": time.perf_counter() - readback_started,
    "quota": quota,
    "authentication": "Existing built-in Kaggle notebook authentication; no credentials printed, copied or created",
    "scope": "API quota, private upload and provider-mounted readback capability only; not independent teammate approval",
}
with (base / "result.json").open("x") as stream:
    json.dump(result, stream, indent=2)
print(json.dumps(result), flush=True)